<a href="https://colab.research.google.com/github/guilhermek32/ListaDataScience-2026.1/blob/main/Lista_1_Antonio_Guilherme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone "https://github.com/guilhermek32/ListaDataScience-2026.1.git"
# Entra na pasta clonada
%cd ListaDataScience-2026.1

Cloning into 'ListaDataScience-2026.1'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 0), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), 2.38 MiB | 4.83 MiB/s, done.


Aluno: Antonio Guilherme da Silva

Nome da base: Medical Appointment No Shows.csv

1.

Faça um código em Python que carregue a base escolhida e gere um diagnóstico
inicial automatizado. Mostre dimensões da base, tipos de dados, valores ausentes,
duplicidades, cardinalidade das colunas e possíveis inconsistências de leitura. Em
seguida, defina qual será o problema computacional tratado no cenário escolhido,
deixando claro se a tarefa será de classificação, regressão ou segmentação. Organize
essa etapa em uma função reutilizável.

In [2]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)
FILE_PATH = "Medical Appointment No Shows.csv"

In [5]:
path = Path(FILE_PATH)
if not path.exists():
    raise FileNotFoundError(f"File not found: {path.resolve()}")
# Try common encodings
encodings_to_try = ["utf-8-sig", "utf-8", "latin-1", "cp1252"]
text_sample = None
used_encoding = None
for enc in encodings_to_try:
    try:
        text_sample = path.read_text(encoding=enc, errors="strict")
        used_encoding = enc
        break
    except UnicodeDecodeError:
        continue
if text_sample is None:
    # fallback with replacement to avoid blocking
    used_encoding = "utf-8"
    text_sample = path.read_text(encoding=used_encoding, errors="replace")
sample = "\n".join(text_sample.splitlines()[:50])
try:
    delimiter = csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
except Exception:
    delimiter = ","
df = pd.read_csv(path, sep=delimiter, encoding=used_encoding)
print(f"Loaded: {path.name}")
print(f"Encoding: {used_encoding}")
print(f"Delimiter: {repr(delimiter)}")
print(f"Shape: {df.shape}")
df.head()
df.shape

Loaded: Medical Appointment No Shows.csv
Encoding: utf-8-sig
Delimiter: ','
Shape: (110527, 14)


(110527, 14)

In [6]:
print("== Dimensions ==")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print("\n== Data types ==")
display(df.dtypes.rename("dtype").to_frame())
print("\n== Column names ==")
print(list(df.columns))
# Check if any column names have leading/trailing spaces
col_strip_issues = [c for c in df.columns if c != c.strip()]
print("\nHeader strip issues:", col_strip_issues if col_strip_issues else "None")

== Dimensions ==
Rows: 110,527
Columns: 14

== Data types ==


,dtype
PatientId,float64
AppointmentID,int64
Gender,str
ScheduledDay,str
AppointmentDay,str
Age,int64
Neighbourhood,str
Scholarship,int64
Hipertension,int64
Diabetes,int64



== Column names ==
['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay', 'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No-show']

Header strip issues: None


In [12]:
issues = {}
# ---------------------------
# A) Missing/duplicates/cardinality base tables
# ---------------------------
null_tokens = {"", "na", "n/a", "null", "none", "nan", "?"}
df_str = df.copy()
for col in df_str.columns:
    if pd.api.types.is_string_dtype(df_str[col]) or df_str[col].dtype == "object":
        df_str[col] = df_str[col].astype(str).str.strip()
        df_str[col] = df_str[col].replace({tok: np.nan for tok in null_tokens})
missing_count = df_str.isna().sum().sort_values(ascending=False)
missing_pct = (missing_count / len(df_str) * 100).round(2)
missing_report = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": missing_pct
})
cardinality_report = (
    df_str.nunique(dropna=True)
    .sort_values(ascending=False)
    .rename("n_unique")
    .to_frame()
)
# ---------------------------
# B) Inconsistencies checks
# ---------------------------
# 1) Whitespace inconsistencies in text columns
obj_cols = [c for c in df.columns if pd.api.types.is_string_dtype(df[c]) or df[c].dtype == "object"]
whitespace_issues = {}
for c in obj_cols:
    s = df[c].astype(str)
    whitespace_issues[c] = int((s != s.str.strip()).sum())
issues["whitespace_issues"] = {k: v for k, v in whitespace_issues.items() if v > 0}
# 2) Datetime parsing checks
for dt_col in ["ScheduledDay", "AppointmentDay"]:
    if dt_col in df.columns:
        parsed = pd.to_datetime(df[dt_col], errors="coerce", utc=True)
        issues[f"{dt_col}_invalid_datetime"] = int(parsed.isna().sum())
# 3) Temporal logic check
if {"ScheduledDay", "AppointmentDay"}.issubset(df.columns):
    sched = pd.to_datetime(df["ScheduledDay"], errors="coerce", utc=True)
    appt = pd.to_datetime(df["AppointmentDay"], errors="coerce", utc=True)
    valid_pair = ~(sched.isna() | appt.isna())
    issues["appointment_before_scheduled"] = int(((appt < sched) & valid_pair).sum())
    issues["appointment_equals_scheduled"] = int(((appt == sched) & valid_pair).sum())
    issues["appointment_distinct_time_values"] = int(appt.dt.strftime("%H:%M:%S").dropna().nunique())
# 4) Numeric plausibility checks
if "Age" in df.columns:
    age = pd.to_numeric(df["Age"], errors="coerce")
    issues["age_non_numeric"] = int(age.isna().sum())
    issues["age_negative"] = int((age < 0).sum())
    issues["age_gt_120"] = int((age > 120).sum())
# 5) Domain checks for key binary/categorical columns
candidate_binary_cols = ["Scholarship", "Hipertension", "Diabetes", "Alcoholism", "SMS_received", "No-show", "Gender"]
domain_report = {}
for c in candidate_binary_cols:
    if c in df.columns:
        vals = (
            df[c]
            .dropna()
            .astype(str)
            .str.strip()
            .replace("", np.nan)
            .dropna()
            .unique()
            .tolist()
        )
        domain_report[c] = sorted(vals)
print("== Domain preview for key categorical/binary columns ==")
display(pd.DataFrame({
    "column": list(domain_report.keys()),
    "observed_values": [domain_report[k] for k in domain_report.keys()]
}))
print("\n== Potential inconsistencies ==")
for k, v in issues.items():
    print(f"- {k}: {v}")
# ---------------------------
# C) Final compact report
# ---------------------------
summary = pd.DataFrame({
    "metric": [
        "n_rows",
        "n_columns",
        "n_duplicate_full_rows",
        "columns_with_missing",
        "total_missing_cells",
    ],
    "value": [
        int(df.shape[0]),
        int(df.shape[1]),
        int(df_str.duplicated().sum()),
        int((df_str.isna().sum() > 0).sum()),
        int(df_str.isna().sum().sum()),
    ]
})
display(summary)
print("\nTop cardinality columns:")
display(cardinality_report.head(10))
print("\nColumns with missing values (>0):")
display(missing_report[missing_report["missing_count"] > 0])

== Domain preview for key categorical/binary columns ==


,column,observed_values
0,Scholarship,"[0, 1]"
1,Hipertension,"[0, 1]"
2,Diabetes,"[0, 1]"
3,Alcoholism,"[0, 1]"
4,SMS_received,"[0, 1]"
5,No-show,"[No, Yes]"
6,Gender,"[F, M]"



== Potential inconsistencies ==
- whitespace_issues: {}
- ScheduledDay_invalid_datetime: 0
- AppointmentDay_invalid_datetime: 0
- appointment_before_scheduled: 38568
- appointment_equals_scheduled: 0
- appointment_distinct_time_values: 1
- age_non_numeric: 0
- age_negative: 1
- age_gt_120: 0


,metric,value
0,n_rows,110527
1,n_columns,14
2,n_duplicate_full_rows,0
3,columns_with_missing,0
4,total_missing_cells,0



Top cardinality columns:


,n_unique
AppointmentID,110527
ScheduledDay,103549
PatientId,62299
Age,104
Neighbourhood,81
AppointmentDay,27
Handcap,5
Gender,2
Hipertension,2
Scholarship,2



Columns with missing values (>0):


,missing_count,missing_pct
